# Analysis and Visualization of Complex Agro-Environmental Data
---
## Descriptive statistics

As an example we will work on a subset of a database that resulted from integrating information from several river fish biomonitoring programmes accross Europe. This subset includes data for some Mediterranean countries. Each case (rows) corresponds to a fish sampling point. Variables (columns) includes coordinates, country and catchment identifiers, local scale environmental variables, climatic variables, human pressures and fish presence/absence data.

When working with a new dataset, one of the most useful things to do is to begin to visualize the data. By using tables, histograms, box plots, and other visual tools, we can get a better idea of what the data may be trying to tell us, and we can gain insights into the data that we may have not discovered otherwise.

We will be going over how to perform some basic visualisations in Python, and, most importantly, we will learn how to begin exploring data from a graphical perspective.

In [ ]:
import pandas as pd
import zipfile
import seaborn as sns # For plotting
import matplotlib.pyplot as plt # For showing plots
import numpy as np

#### Import, visualize and summarize table properties

In [ ]:
df = pd.read_csv('EFIplus_medit.zip',compression='zip', sep=";")

In [ ]:
print(df)

In [ ]:
df.head(5)

In [ ]:
df.info()

In [ ]:
list(df.columns)

#### Clean and reajust the dataset

In [ ]:
# clean up the dataset to remove unnecessary columns (eg. REG) 
df.drop(df.iloc[:,5:15], axis=1, inplace=True) # axis=1 - columns; inplace=True - the changes will be saved to the original data frame. 

# let's rename some columns so that they make sense
df.rename(columns={'Sum of Run1_number_all':'Total_fish_individuals'}, inplace=True) # inplace="True" means that df will be updated

# for consistency, let's also make all column labels of type string
df.columns = list(map(str, df.columns))

In [ ]:
# Check data types
pd.options.display.max_rows = 154 # maximum number of rows displayed.
df.dtypes

In [ ]:
# Number of values per variable
df.count()

### Handling missing data

In [ ]:
# Number of missing values (NaN) per variable
df.isnull().sum()

Let's drop the rows (records or observations) that contains at least one missing value (this is the default of the dropna method - other inactive examples are given below).

In [ ]:
df2 = df.dropna() # drops rows when at least one element is a missing value
df2.info()

#df2 = df.dropna(how='all') # drops rows when all elements are missing values
#df2.info()

#df2 = df.dropna(how='all', axis=1) # drops columns when at least one element is a missing value
#df2.info()

### Numerical summaries

In [ ]:
# mean and median (rounded to 2 decimal cases)
mean = round(df2['prec_ann_catch'].mean(), 2)
median = round(df2['prec_ann_catch'].median(), 2)
print(mean, median)

In [ ]:
# the catchment with more data
print(df2['Catchment_name'].mode())

In [ ]:
# A fast way of getting a summary statistics of quantitative data (int or float)

# before dropping NaNs (rounded to 2 decimal cases)
round(df.describe() ,2) 

In [ ]:
# after dropping NaNs (rounded to 2 decimal cases)
round(df2.describe() ,2) 

In [ ]:
# Retrieving the number of observation for each category of a categorical variable (eg. Country)
country_count = pd.crosstab(index = df2['Country'], columns='count')
print(country_count)

In [ ]:
# Retrieving the number of observation for each catchment
catchment_count = pd.crosstab(index = df2['Catchment_name'], columns='count')
print(catchment_count)

### Plotting qualitative data

Check here: https://seaborn.pydata.org/generated/seaborn.catplot.html

##### Barplots (categorical plots)

Plotting the number of sites per country

In [ ]:
country_count.index

In [ ]:
# Using matplotlib
# Plot a bar chart using the country counts, setting the bar color to sky blue
plt.bar(country_count.index, country_count['count'], color='skyblue')

# Set the label for the x-axis
plt.xlabel('Country')

# Set the label for the y-axis
plt.ylabel('Count')

# Set the title of the plot
plt.title('Number of observations per Country')

# Display the plot on the screen
plt.show()

In [ ]:
# using pandas 'plot' method
country_count.plot(kind='bar') 
plt.show()

In [ ]:
# getting help on the 'plot' method
help(country_count.plot)

In [ ]:
# Fine tuning pandas 'plot' method
country_count.plot(kind='bar', 
                   legend=False, 
                   color='skyblue', 
                   title='Number of observations per Country', 
                   xlabel='Country', 
                   ylabel='Count', 
                   rot=0) # rot=0 - rotation of x labels
plt.show()

In [ ]:
# Now using seaborn
sns.catplot(x="Country", data=df2, kind="count", color="skyblue")
plt.show()

In [ ]:
# Same thing but now changing the order and including a title:
sns.catplot(x="Country", data=df2, kind="count", color="skyblue", order=['Italy', 'Portugal', 'Spain']).set(title='Number of observations per Country')
plt.show()

# Another way of including a title:
# ax = sns.catplot(x="Country", data=df, kind="count", color="skyblue", order=['Italy', 'Portugal', 'Spain'])
# ax.set_titles('Number of observations per Country')
# plt.show()

Plotting the number of sites per catchment

In [ ]:
# using Seaborn
sns.catplot(x="Catchment_name", data=df2, kind="count", color="skyblue")
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Same thing but in a decreasing order:
order = df2["Catchment_name"].value_counts().index # the order of the categories is defined by the number of observations (decreasing order)

sns.catplot(x="Catchment_name", data=df2, kind="count", color="skyblue", order=order)
plt.xticks(rotation=90)
plt.show()

##### Pie charts

In [ ]:
# pie chart
colors = sns.color_palette('pastel')
labels = list(country_count.index) # list of country names
plt.pie(list(country_count.iloc[:,0]), labels=labels, colors = colors, autopct = '%0.0f%%')
plt.show()

##### Treemaps

To plot treemaps you'll need to install `squarify`: run `pip install squarify` in the terminal.

In [ ]:
import squarify as sqrf

labels = list(country_count.index) # list of country names

sqrf.plot(sizes=list(country_count.iloc[:,0]), # select all rows from the 1st column of data
          label=labels, # names of countries
          color=sns.color_palette('viridis',n_colors=len(labels)), # color palette
          text_kwargs={'fontsize': 11, 'color':"white"}, # label format
          pad=0.25) # define space between areas
plt.show()


### Plotting quantitative data

#### Strip plots
check here: https://seaborn.pydata.org/generated/seaborn.stripplot.html

In [ ]:
# plot the mean annual total precipitation in the upstream catchment of each site
sns.stripplot(df2['prec_ann_catch'])
plt.show()


#### Histograms
check here: https://seaborn.pydata.org/generated/seaborn.histplot.html

In [ ]:
# histogram of the mean annual total precipitation in the upstream catchment of each site
sns.histplot(df2['prec_ann_catch'], kde = False).set_title("Histogram of precipitation in the upstream catchment")
plt.show()

In [ ]:
# More variations
sns.histplot(
    df["prec_ann_catch"], 
    kde=True,
    stat="density", # plot proportions instead of frequencies
    kde_kws=dict(cut=3),
    alpha=.4, # transparency
    edgecolor=(1, 1, 1, 0.4), # bar contour lines (r, g, b, alpha)
).set_title("Histogram of precipitation in the upstream catchment")
plt.show()

### Bar plots

Check here: https://seaborn.pydata.org/generated/seaborn.barplot.html

In [ ]:
# bar plot of Total Annual Precipitation by country
sns.barplot(x="Country", y="prec_ann_catch", data=df)
plt.show()

In [ ]:
# bar plot of Total Annual Precipitation by catchment
sns.barplot(data=df, x="Catchment_name", y="prec_ann_catch")
plt.xticks(rotation=90)
plt.show()

### Boxplots

Check here: https://seaborn.pydata.org/generated/seaborn.boxplot.html

In [ ]:
# Box plot of Total Annual Precipitation
sns.boxplot(df["prec_ann_catch"]).set_title("Box plot of Total Annual Precipitation")
plt.show()

In [ ]:
# no whiskers (data points outside the box instead)
sns.boxplot(df["prec_ann_catch"], whis=0).set_title("Box plot of Total Annual Precipitation")
plt.show()

In [ ]:
# Box plot of Total Annual Precipitation by country
sns.boxplot(x="Country", y="prec_ann_catch", data=df).set_title("Box plot of Total Annual Precipitation")
plt.show()

In [ ]:
# same thing but only for Portugal
df_port = df[df['Country']=='Portugal']

sns.histplot(
    df_port["prec_ann_catch"], 
    kde=True,
    stat="density",
    kde_kws=dict(cut=3),
    alpha=.4,
    edgecolor=(1, 1, 1, 0.4),
).set_title("Histogram of precipitation in the upstream catchment")
plt.show()


### Violin plots

Check here: https://seaborn.pydata.org/generated/seaborn.violinplot.html

In [ ]:
# violin plot of Total Annual Precipitation by country
sns.violinplot(data=df, y="prec_ann_catch").set_title("Violin plot of Total Annual Precipitation")
plt.show()

### Raincloud plots

Raincloud plots combine violin plots, boxplots and strip plots into a single chart. To plot raincloud plots it is helpful to use the `ptitprince` library. You may need to install `ptitprince`by running `pip install ptitprince` in the terminal.

In [ ]:
import ptitprince as pt

In [ ]:
help(pt.RainCloud)

In [ ]:
# Raincloud plot of Total Annual Precipitation

pt.RainCloud(y="prec_ann_catch", data=df, 
             bw=0.2, # defines how smooth is the distribution curve of the violin plot (cloud)
             width_viol=0.4, # width of the half violin (cloud)
             width_box=0.05, # width of the box
             orient='h', # orientation
             move=0.15, # position of the strip plot
             offset=0,# relative position of the half violin (cloud) in relation to the boxplot
             jitter=0.09) # allows to define the width of the strip plot (rain)

plt.title("Raincloud plot of Total Annual Precipitation")
plt.show()

In [ ]:
# Raincloud plot of Total Annual Precipitation by country

pt.RainCloud(x='Country', y="prec_ann_catch", data=df, 
             bw=0.2, # defines how smooth is the distribution curve of the violin plot (cloud)
             width_viol=1.2, # width of the half violin (cloud)
             width_box=0.15, # wdth of the box
             orient='h', # orientation
             move=0.15, #position of the strip plot
             offset=0,# relative position of the half violin (cloud) in relation to the boxplot
             jitter=0.09) # allows to define the width of the strip plot (rain)

plt.title("Raincloud plot of Total Annual Precipitation")
plt.show()

In [ ]:
# Vertical raincloud plot of Total Annual Precipitation by country

pt.RainCloud(x='Country', y="prec_ann_catch", data=df, 
             bw=0.2, # defines how smooth is the distribution curve of the violin plot (cloud)
             width_viol=1.2, # width of the half violin (cloud)
             width_box=0.08, # wdth of the box
             move=0.15, #position of the strip plot
             offset=0,# relative position of the half violin (cloud) in relation to the boxplot
             jitter=0.08) # allows to define the width of the strip plot (rain)

plt.title("Raincloud plot of Total Annual Precipitation")
plt.show()